# Using Tools
This notebook demonstrates how we can use tools to gather information and provide more accurate responses. We'll build a weather-checking mock as an example.

## What we'll learn:
- Basic interaction with Language Models (LLM)
- How to create and use tools with AI
- The complete flow of an AI agent using tools
- Understanding the message flow in a tool-enabled conversation

## Setup

In [1]:
import json
import os
from dotenv import load_dotenv
from libs.messages import UserMessage, SystemMessage, ToolMessage  # Different message types
from libs.tooling import tool  # Tool decorator for creating AI tools
from libs.llm import LLM  # Our Language Model wrapper

In [3]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")

In [4]:
chat_model = LLM(api_key=api_key, base_url=base_url)

## Basic LLM Interaction
Before we dive into tools, let's understand how to interact with our Language Model in its simplest form.
There are two main ways to communicate with the model:

In [5]:
# Method 1: Simple single-turn query
response = chat_model.invoke("What is an AI Agent?")
print("Single Query Response:\n", response.content)

Single Query Response:
 An AI agent is a system or entity that uses artificial intelligence techniques to perceive its environment, make decisions, and take actions to achieve specific goals. AI agents can operate autonomously or semi-autonomously and can be found in various applications, ranging from simple rule-based systems to complex machine learning models.

Key characteristics of AI agents include:

1. **Perception**: AI agents can gather information from their environment through sensors or data inputs. This could involve processing visual data, audio signals, or other forms of input.

2. **Decision-Making**: Based on the information they perceive, AI agents can analyze the data and make decisions. This can involve reasoning, planning, and learning from past experiences.

3. **Action**: After making decisions, AI agents can take actions to interact with their environment. This could involve physical actions (like a robot moving) or digital actions (like a software agent executin

In [6]:
# Method 2: Multi-turn conversation with specific roles
messages = [
    SystemMessage(content="You're an OpenAI API specialist"),
    UserMessage(content="What is Function Calling?")
]
response = chat_model.invoke(messages)
print("\nStructured Conversation Response:\n", response.content)


Structured Conversation Response:
 Function calling is a programming concept where a function (a block of code designed to perform a specific task) is executed or invoked in a program. When a function is called, the program temporarily transfers control to that function, executes its code, and then returns control back to the point where the function was called. 

### Key Concepts of Function Calling:

1. **Function Definition**: Before a function can be called, it must be defined. This includes specifying the function's name, parameters (if any), and the code that will be executed.

   ```python
   def greet(name):
       return f"Hello, {name}!"
   ```

2. **Function Call**: To execute the function, you use its name followed by parentheses. If the function requires parameters, you provide them within the parentheses.

   ```python
   message = greet("Alice")
   print(message)  # Output: Hello, Alice!
   ```

3. **Parameters and Arguments**: Functions can take parameters, which are v

## Building an AI Tool
Now let's make our AI more capable by giving it a tool to check the weather.
This demonstrates how we can extend AI capabilities beyond just conversation.

### Understanding the Tool Structure:
1. We use the `@tool` decorator to mark a function as an AI tool
2. The tool needs clear documentation and typed parameters
3. The tool should return structured data

In [7]:
@tool
def get_weather(city: str):
    """Get the current temperature for a city.

    Args:
        city (str): Name of the city to check weather for

    Returns:
        dict: Contains temperature information for the requested city
    """
    # In a real application, this would call a weather API
    mock_weather = {
        "São Paulo": "28°C",
        "Oslo": "-3°C",
        "New York": "15°C",
        "Tokyo": "22°C",
        "Hyderabad":"36°C"
    }
    return {"temperature": mock_weather.get(city, "Unknown")}

In [12]:
@tool
def get_current_projects(year:str):
    """Get the current temperature for a city.

    Args:
        year (str): Year to check projects build on the year

    Returns:
        dict: Contains temperature information for the requested city
    """
    # In a real application, this would query a project management system
    projects = {
        "2017": "DSM",
        "2018": "Vision",
        "2019": "NLP",
        "2020": "Reinforcement Learning",
        "2021": "Multimodal AI",
        "2022": "Generative AI",
        "2023": "AI Agents",
        "2024": "AI Agents",
        "2025": "AI Agents",
        "2026": "AI Agents",
        "2027": "AI Agents",
    }
    return {"project": projects.get(year, "Unknown")}

In [13]:
# Bind the tool to an LLM
chat_model_with_tools = LLM(tools=[get_weather, get_current_projects])

## Understanding the Tool Usage Flow
Let's break down how the AI uses tools step by step:

1. User asks a question about weather
2. AI recognizes the need to use the weather tool
3. AI makes a tool call
4. Tool executes and returns results
5. AI processes the tool's response
6. AI formulates a natural language response

Let's see this in action:

In [14]:
# Set up our system with clear instructions
messages = [
    SystemMessage(
        content="You are a helpful assistant that can access a tool to get current temperature "
                "for cities. Use the tool whenever someone asks about the weather or temperature "
                "in a specific location. Infor the user if you don't know the answer."
    ),
    UserMessage(content="How cold is it in Oslo?")
]

In [24]:
messages = [
    SystemMessage(
        content="You are a helpful assistant that can access a tool to get projects that developers worked on based on year."
                "Use the tool whenever someone asks about the projects "
                "in a specific year. Inform the user if you don't know the answer."
    ),
    UserMessage(content="What is the current project build in 2024?")
]

In [25]:
# AI decides to use the weather tool
ai_message = chat_model_with_tools.invoke(messages)
ai_message

AIMessage(content=None, role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ds3x07gHXcpH9Y2KweMux76W', function=Function(arguments='{"year":"2024"}', name='get_current_projects'), type='function')])

In [26]:
# Check messages structure
messages.append(ai_message)
messages

[SystemMessage(content="You are a helpful assistant that can access a tool to get projects that developers worked on based on year.Use the tool whenever someone asks about the projects in a specific year. Inform the user if you don't know the answer.", role='system'),
 UserMessage(content='What is the current project build in 2024?', role='user'),
 AIMessage(content=None, role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ds3x07gHXcpH9Y2KweMux76W', function=Function(arguments='{"year":"2024"}', name='get_current_projects'), type='function')])]

In [27]:
# Tool call id will be required later when creating the ToolMessage
tool_call_id = messages[-1].tool_calls[0].id
tool_call_id

'call_ds3x07gHXcpH9Y2KweMux76W'

In [28]:
# Extract the arguments
args = json.loads(messages[-1].tool_calls[0].function.arguments)
args

{'year': '2024'}

In [19]:
# Execute the tool with the AI's requested parameters
tool_result = get_weather(**args)
tool_result

{'temperature': '-3°C'}

In [20]:
# Create a tool response message
tool_message = ToolMessage(
    content=tool_result["temperature"],
    tool_call_id=tool_call_id,
    name="get_weather"
)
tool_message

ToolMessage(content='-3°C', role='tool', tool_call_id='call_HnvIpKGwHIi8Fu1Uex6csvJV', name='get_weather')

In [21]:
# Check messages structure
messages.append(tool_message)
messages

[SystemMessage(content="You are a helpful assistant that can access a tool to get current temperature for cities. Use the tool whenever someone asks about the weather or temperature in a specific location. Infor the user if you don't know the answer.", role='system'),
 UserMessage(content='How cold is it in Oslo?', role='user'),
 AIMessage(content=None, role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_HnvIpKGwHIi8Fu1Uex6csvJV', function=Function(arguments='{"city":"Oslo"}', name='get_weather'), type='function')]),
 ToolMessage(content='-3°C', role='tool', tool_call_id='call_HnvIpKGwHIi8Fu1Uex6csvJV', name='get_weather')]

In [22]:
# Let AI formulate final response
ai_message = chat_model_with_tools.invoke(messages)
ai_message

AIMessage(content='The current temperature in Oslo is -3°C.', role='assistant', tool_calls=None)

In [23]:
# Check messages structure
messages.append(ai_message)
messages

[SystemMessage(content="You are a helpful assistant that can access a tool to get current temperature for cities. Use the tool whenever someone asks about the weather or temperature in a specific location. Infor the user if you don't know the answer.", role='system'),
 UserMessage(content='How cold is it in Oslo?', role='user'),
 AIMessage(content=None, role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_HnvIpKGwHIi8Fu1Uex6csvJV', function=Function(arguments='{"city":"Oslo"}', name='get_weather'), type='function')]),
 ToolMessage(content='-3°C', role='tool', tool_call_id='call_HnvIpKGwHIi8Fu1Uex6csvJV', name='get_weather'),
 AIMessage(content='The current temperature in Oslo is -3°C.', role='assistant', tool_calls=None)]

In [30]:
tool_result = get_current_projects(**args)
tool_result

{'project': 'AI Agents'}

In [31]:
# Create a tool response message
tool_message = ToolMessage(
    content=tool_result["project"],
    tool_call_id=tool_call_id,
    name="get_current_projects"
)
tool_message

ToolMessage(content='AI Agents', role='tool', tool_call_id='call_ds3x07gHXcpH9Y2KweMux76W', name='get_current_projects')

In [32]:
# Check messages structure
messages.append(tool_message)
messages

[SystemMessage(content="You are a helpful assistant that can access a tool to get projects that developers worked on based on year.Use the tool whenever someone asks about the projects in a specific year. Inform the user if you don't know the answer.", role='system'),
 UserMessage(content='What is the current project build in 2024?', role='user'),
 AIMessage(content=None, role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ds3x07gHXcpH9Y2KweMux76W', function=Function(arguments='{"year":"2024"}', name='get_current_projects'), type='function')]),
 ToolMessage(content='AI Agents', role='tool', tool_call_id='call_ds3x07gHXcpH9Y2KweMux76W', name='get_current_projects')]

In [33]:
# Let AI formulate final response
ai_message = chat_model_with_tools.invoke(messages)
ai_message

AIMessage(content='The current project built in 2024 is "AI Agents."', role='assistant', tool_calls=None)

In [34]:
# Check messages structure
messages.append(ai_message)
messages

[SystemMessage(content="You are a helpful assistant that can access a tool to get projects that developers worked on based on year.Use the tool whenever someone asks about the projects in a specific year. Inform the user if you don't know the answer.", role='system'),
 UserMessage(content='What is the current project build in 2024?', role='user'),
 AIMessage(content=None, role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ds3x07gHXcpH9Y2KweMux76W', function=Function(arguments='{"year":"2024"}', name='get_current_projects'), type='function')]),
 ToolMessage(content='AI Agents', role='tool', tool_call_id='call_ds3x07gHXcpH9Y2KweMux76W', name='get_current_projects'),
 AIMessage(content='The current project built in 2024 is "AI Agents."', role='assistant', tool_calls=None)]